In [ ]:
import coflandscaper as cl

### Before you run this notebook (required)

- Place exactly one node XYZ in 0_node/ and one linker XYZ in 0_linker/.
- At the connection points, insert a dummy atom: **He** if node and linker are connected by a single bond and **Se** f node and linker are connected by a double bond .
- If your COF contains Se atoms, this workflow will not work with double bonds.
- This workflow is in general not tested for metal-containing COFs or MOFs.
- Choose a unique `COF_NAME`; all outputs will be saved in a folder named after the COF.

### Settings & options (what you can choose)

- `TOPOLOGY`: "hcb" or "sql".
- `BOND_TYPE`: "single" or "double" (must match your dummy atom choice).
- `COF_NAME`: "example-cof".
- `MODE`: choose what stacking modes to investigate: "incl" (inclined), "serr" (serrated), or "both".

In [2]:
TOPOLOGY = "hcb"
BOND_TYPE = "single" 
COF_NAME = "cof-1"
MODE = 'both'

### Build single-layer COF + Pre-Optimization:

Creates the single-layer COF and then pre-optimizes it with MACE.


In [ ]:
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, bond_type=BOND_TYPE, cof_name=COF_NAME)
preopt = cl.MacePreopt()
preopt.run(cof_name=COF_NAME)

### Create ILD×ILS structure-matrix

Generate stacking variants by changing interlayer distance (ILD, along $z$) and interlayer slipping (ILS, in-plane).

Defaults:
- Default max. slip length/angle correspond to AB stacking and are auto-computed.
- Default ILD range is 3.0 to 4.5 in 0.1 A steps.
- Default ILS range is 0 to max in 1 A steps.

In [ ]:
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

### MACE single-point energies

Compute locally on this machine single-point energies for each structure to build the energy landscape. No geometry optimization is performed.

In [ ]:
sp = cl.MaceSP()
sp.run_mode(cof_name=COF_NAME, mode=MODE)

### Potential Energy Landscape (PES)

The PES is an approximation that treats each structure as identical except for its `ILD`/`ILS` values. Under this assumption, the landscape provides a reasonable qualitative map of relative energies across stacking configurations. The goal is to identify candidate minima that can then be validated and refined with full geometry optimization.

In [ ]:
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE)

Add extra (ILD, ILS) points you want to include in the selection. Use a list of tuples in Angstrom: [(ILD, ILS), ...]. These are added on top of the automatically detected minima for each mode (serr/incl).

In [ ]:
EXTRA_SERR = [(3.6, 0.0)]
EXTRA_INCL = [(3.6, 0.0)]

Selection copies CIFs at the auto-detected minima (and any extra ILD/ILS pairs you provide) into the selection folders for downstream optimization.

In [ ]:
selector = cl.SelectCofs()
selector.run_mode(cof_name=COF_NAME, mode=MODE, selections_serr=EXTRA_SERR, selections_incl=EXTRA_INCL)

### MACE geometry optimization

Perform MACE relaxations of the selected stacking structures to locate locally optimized geometries and refine relative energies prior to higher-level validation.

In [ ]:
# Defaults
opt = cl.MaceFullOpt()
opt.run_mode(cof_name=COF_NAME, mode=MODE)

### Analyze + Visualize

`analyze` computes ILD/ILS for the final optimized structures and writes `final_structures.csv` in the output folder (defaults to `COF_NAME`/4_{`COF_NAME`}_final_structures).

In [ ]:
# Defaults
analyze = cl.analyze
analyze(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Defaults
visualize = cl.visualizecof
visualize(cof_name=COF_NAME, mode=MODE)